<a href="https://colab.research.google.com/github/guanhaibin/DataMining/blob/master/Fine_tune_EMS2_Embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# # !pip -q install kaggle

# from google.colab import files
# files.upload()

# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json # files.upload()

In [13]:
# import json, os

# with open("/root/.kaggle/kaggle.json", "r") as f:
#     creds = json.load(f)

# os.environ["KAGGLE_USERNAME"] = creds["username"]
# os.environ["KAGGLE_KEY"] = creds["key"]

# print("Set KAGGLE_USERNAME and KAGGLE_KEY")


In [16]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
# kagglehub.login()


In [19]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

# base_path = kagglehub.competition_download('cafa-6-protein-function-prediction')

# print('Data source import complete.')

# os.listdir(base_path)

In [20]:
import kagglehub, os, glob

base_path = kagglehub.competition_download("cafa-6-protein-function-prediction")
print("Downloaded to:", base_path)

# show what’s inside
print("Top level:", os.listdir(base_path))

# show all files (recursive) so you can see exact names/structure
all_files = glob.glob(base_path + "/**/*", recursive=True)
for f in all_files[:40]:
    print(f)
print("Total files:", len(all_files))


Downloaded to: /root/.cache/kagglehub/competitions/cafa-6-protein-function-prediction
Top level: ['IA.tsv', 'Test', 'sample_submission.tsv', 'Train']
/root/.cache/kagglehub/competitions/cafa-6-protein-function-prediction/IA.tsv
/root/.cache/kagglehub/competitions/cafa-6-protein-function-prediction/Test
/root/.cache/kagglehub/competitions/cafa-6-protein-function-prediction/sample_submission.tsv
/root/.cache/kagglehub/competitions/cafa-6-protein-function-prediction/Train
/root/.cache/kagglehub/competitions/cafa-6-protein-function-prediction/Test/testsuperset.fasta
/root/.cache/kagglehub/competitions/cafa-6-protein-function-prediction/Test/testsuperset-taxon-list.tsv
/root/.cache/kagglehub/competitions/cafa-6-protein-function-prediction/Train/train_taxonomy.tsv
/root/.cache/kagglehub/competitions/cafa-6-protein-function-prediction/Train/train_sequences.fasta
/root/.cache/kagglehub/competitions/cafa-6-protein-function-prediction/Train/train_terms.tsv
/root/.cache/kagglehub/competitions/caf

In [21]:
# !pip install -U protobuf==3.20.3
# !pip install biopython > dev null
# !pip install obonet > dev nul
# !pip install torchmetrics

In [22]:
import os
import copy
import random
import pandas as pd
import numpy as np
from tqdm import tqdm
import time
import matplotlib.pyplot as plt
plt.style.use('ggplot')
from Bio import SeqIO
# TORCH MODULES FOR METRICS COMPUTATION :
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torch import nn
from torch.utils.data import random_split
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchmetrics.classification import MultilabelF1Score
from torchmetrics.classification import MultilabelAccuracy

from transformers import AutoTokenizer, AutoModel
from transformers import DataCollatorWithPadding

In [23]:
class CFG:
    DATA_DIR = base_path  # <- this is the key change

    TRAIN_TERMS_PATH     = os.path.join(DATA_DIR, "Train", "train_terms.tsv")
    TRAIN_SEQUENCES_PATH = os.path.join(DATA_DIR, "Train", "train_sequences.fasta")
    TRAIN_TAXONOMY_PATH  = os.path.join(DATA_DIR, "Train", "train_taxonomy.tsv")

    TEST_SEQUENCES_PATH  = os.path.join(DATA_DIR, "Test", "testsuperset.fasta")
    IA_PATH              = os.path.join(DATA_DIR, "IA.tsv")
    TEST_TAXON_PATH      = os.path.join(DATA_DIR, "Test", "testsuperset-taxon-list.tsv")
    OBO_PATH             = os.path.join(DATA_DIR, "Train", "go-basic.obo")

    # EDA & Plotting
    COLORS = ['#221f1f', '#b20710', '#e50914', 'grey']
    BACKGROUND_COLOR = '#f5f6f6'
    ESM2_MODEL_NAME = 'facebook/esm2_t6_8M_UR50D'
    ESM3_MODEL_NAME = 'esm3_sm_open_v1'

    batch_size =16
    lr = 1e-5
    # Debug toggles (set to None for full run)
    DEBUG_TRAIN_N = 4000          # use first N train proteins
    DEBUG_TEST_N = 200            # predict first N test proteins
    DEBUG_POS_WEIGHT_BATCHES = 50 # estimate pos_weight from first N batches
    DEBUG_THRESHOLD_GRID = np.linspace(0.01, 0.50, 25)  # faster threshold search

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(CFG.device)
device=CFG.device
MAX_LENGTH = 1022

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

cuda


In [3]:
def normalize_fasta_id(raw_id: str) -> str:
    # "sp|A0A0C5B5G6|MOTSC_HUMAN" -> "A0A0C5B5G6"
    s = raw_id.strip()
    if "|" in s:
        parts = s.split("|")
        if len(parts) >= 2:
            return parts[1].strip()
    return s.split()[0].strip()


In [4]:
import scipy.sparse as sp
from collections import defaultdict
from sklearn.metrics import average_precision_score

# --- Your Evaluator class (unchanged) ---
# (Use the Evaluator you pasted; not repeating here to save space.)
# Make sure it's defined above build_evaluator_from_train_terms().

def load_go_ancestors(obo_path):
    """Parse go-basic.obo and return ancestors[GO_ID] = set(all ancestors)."""
    parents = {}
    current_term = None

    with open(obo_path, "r") as f:
        for line in f:
            line = line.strip()

            if line == "[Term]":
                current_term = None
            elif line.startswith("id: GO:"):
                current_term = line.split("id: ")[1]
                parents[current_term] = set()
            elif line.startswith("is_a: GO:") and current_term is not None:
                parent = line.split("is_a: ")[1].split(" !")[0]
                parents[current_term].add(parent)

    ancestors = {}

    def dfs(term, visited):
        if term in ancestors:
            return ancestors[term]
        result = set()
        for p in parents.get(term, []):
            if p not in visited:
                visited.add(p)
                result.add(p)
                result |= dfs(p, visited)
        return result

    for term in parents:
        ancestors[term] = dfs(term, set())

    return ancestors


def build_ancestor_matrix(go_terms_list, ancestors):
    """A[i, j] = 1 if go_terms_list[j] is an ancestor of go_terms_list[i]."""
    n = len(go_terms_list)
    term_to_idx = {t: i for i, t in enumerate(go_terms_list)}
    A = np.zeros((n, n), dtype=np.bool_)

    for child_idx, term in enumerate(go_terms_list):
        for anc in ancestors.get(term, []):
            if anc in term_to_idx:
                A[child_idx, term_to_idx[anc]] = True

    return A


def get_top_terms_from_train_terms(train_terms_path, term_type, num_labels):
    df = pd.read_csv(train_terms_path, sep="\t")

    # Map term_type to CAFA6 aspect codes
    aspect_map = {"BP": "P", "MF": "F", "CC": "C", "all": None}

    if term_type not in aspect_map:
        raise ValueError(f"term_type must be one of {list(aspect_map.keys())}, got {term_type}")

    code = aspect_map[term_type]
    if code is not None:
        df = df[df["aspect"] == code]

    top_terms = (
        df.groupby("term")["EntryID"]
          .count()
          .sort_values(ascending=False)
          .head(num_labels)
          .index
          .tolist()
    )
    return top_terms


def build_evaluator_from_train_terms(train_terms_path, term_type, num_labels, ia_path, obo_path):
    # top terms from TSV
    go_terms_list = get_top_terms_from_train_terms(train_terms_path, term_type, num_labels)

    # IA weights
    ia_df = pd.read_csv(ia_path, sep='\t', names=['term', 'ia'],header=None)
    ia_weights = dict(zip(ia_df["term"], ia_df["ia"]))

    # ancestors
    ancestors = load_go_ancestors(obo_path)
    A = build_ancestor_matrix(go_terms_list, ancestors)

    evaluator = Evaluator(
        ia_weights=ia_weights,
        go_terms=go_terms_list,
        ancestor_matrix=A
    )

    return evaluator, go_terms_list

class Evaluator:
    def __init__(self, ia_weights, go_terms, ancestor_matrix=None):
        self.go_terms = list(go_terms)
        self.weights = np.array([ia_weights.get(t, 1.0) for t in self.go_terms], dtype=np.float32)
        self.A = ancestor_matrix  # shape (T, T) boolean

    def _dense_bool(self, y_true):
        if sp.issparse(y_true):
            return y_true.astype(np.bool_).toarray()
        return (y_true > 0.5)

    def _ancestor_closure_bool(self, y):
        if self.A is None:
            return y
        # y boolean (N,T), A boolean (T,T)
        return np.logical_or(y, y @ self.A)

    def _ancestor_closure_prob(self, y):
        if self.A is None:
            return y
        # propagate probs to ancestors
        return np.maximum(y, y @ self.A)

    def _f_from_prec_rec(self, p, r, eps=1e-12):
        return (2.0 * p * r) / (p + r + eps)

    def fmax(self, y_true, y_prob, thresholds):
        y_true_b = self._dense_bool(y_true)
        best_t, best_f = 0.5, -1.0

        for t in thresholds:
            y_pred_b = (y_prob >= t)

            tp = np.logical_and(y_true_b, y_pred_b).sum(axis=1).astype(np.float32)
            fp = np.logical_and(~y_true_b, y_pred_b).sum(axis=1).astype(np.float32)
            fn = np.logical_and(y_true_b, ~y_pred_b).sum(axis=1).astype(np.float32)

            has_pred = (tp + fp) > 0
            has_true = (tp + fn) > 0

            p = (tp[has_pred] / (tp[has_pred] + fp[has_pred] + 1e-12)).mean() if has_pred.any() else 0.0
            r = (tp[has_true] / (tp[has_true] + fn[has_true] + 1e-12)).mean() if has_true.any() else 0.0

            f = self._f_from_prec_rec(float(p), float(r))
            if f > best_f:
                best_f, best_t = f, float(t)

        return best_t, best_f

    def ia_fmax(self, y_true, y_prob, thresholds):
        y_true_b = self._dense_bool(y_true)
        w = self.weights[None, :]  # (1,T)

        best_t, best_f = 0.5, -1.0
        for t in thresholds:
            y_pred_b = (y_prob >= t)

            tp_w = (np.logical_and(y_true_b, y_pred_b) * w).sum(axis=1).astype(np.float32)
            fp_w = (np.logical_and(~y_true_b, y_pred_b) * w).sum(axis=1).astype(np.float32)
            fn_w = (np.logical_and(y_true_b, ~y_pred_b) * w).sum(axis=1).astype(np.float32)

            has_pred = (tp_w + fp_w) > 0
            has_true = (tp_w + fn_w) > 0

            p = (tp_w[has_pred] / (tp_w[has_pred] + fp_w[has_pred] + 1e-12)).mean() if has_pred.any() else 0.0
            r = (tp_w[has_true] / (tp_w[has_true] + fn_w[has_true] + 1e-12)).mean() if has_true.any() else 0.0

            f = self._f_from_prec_rec(float(p), float(r))
            if f > best_f:
                best_f, best_t = f, float(t)

        return best_t, best_f

    def micro_aupr(self, y_true, y_prob):
        y_true_b = self._ancestor_closure_bool(self._dense_bool(y_true))
        y_prob_c = self._ancestor_closure_prob(y_prob)
        re


In [5]:
# ============================================================
# 1) Build top-K labels from train_terms.tsv
# ============================================================
def build_topk_labels_from_train_terms(train_terms_path, term_type, num_labels):
    df = pd.read_csv(train_terms_path, sep="\t")

    aspect_map = {"BP": "P", "MF": "F", "CC": "C", "all": None}
    code = aspect_map.get(term_type, None)
    if term_type != "all":
        if code is None:
            raise ValueError(f"Unknown term_type={term_type}")
        df = df[df["aspect"] == code]

    top_terms = (
        df.groupby("term")["EntryID"]
          .count()
          .sort_values(ascending=False)
          .head(num_labels)
          .index
          .tolist()
    )
    term_to_idx = {t: i for i, t in enumerate(top_terms)}

    df_sub = df[df["term"].isin(top_terms)]
    id_to_terms = df_sub.groupby("EntryID")["term"].apply(list).to_dict()

    id_to_vec = {}
    for entry_id, terms in id_to_terms.items():
        v = np.zeros(len(top_terms), dtype=np.float32)
        for t in terms:
            v[term_to_idx[t]] = 1.0
        id_to_vec[entry_id] = v

    return top_terms, id_to_vec



In [6]:
# ============================================================
# 2) Dataset: FASTA + TSV labels
# ============================================================
class GOFineTuneDatasetFromTSV(Dataset):
    def __init__(self, fasta_path, tokenizer, train_terms_path, term_type, num_labels, max_length=1022):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.records = list(SeqIO.parse(fasta_path, "fasta"))
        if hasattr(CFG, "DEBUG_TRAIN_N") and CFG.DEBUG_TRAIN_N is not None:
            self.records = self.records[:CFG.DEBUG_TRAIN_N]
        self.entry_ids = [normalize_fasta_id(rec.id) for rec in self.records]



        self.top_terms, self.id_to_vec = build_topk_labels_from_train_terms(
            train_terms_path=train_terms_path,
            term_type=term_type,
            num_labels=num_labels
        )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        entry_id = self.entry_ids[idx]
        seq = str(rec.seq)[: self.max_length]

        enc = self.tokenizer(
            seq,
            return_tensors="pt",
            padding=False,
            truncation=True,
            max_length=self.max_length
        )

        y = self.id_to_vec.get(entry_id, np.zeros(len(self.top_terms), dtype=np.float32))

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.from_numpy(y),
            "protein_id": entry_id
        }



In [7]:
# ============================================================
# 3) Collate function: pad and stack labels
# ============================================================
def make_collate_fn(tokenizer):
    collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

    def collate_fn(batch):
        labels = torch.stack([x["labels"] for x in batch], dim=0)
        protein_ids = [x["protein_id"] for x in batch]

        batch_wo = [{"input_ids": x["input_ids"], "attention_mask": x["attention_mask"]} for x in batch]
        padded = collator(batch_wo)

        return {
            "input_ids": padded["input_ids"],
            "attention_mask": padded["attention_mask"],
            "labels": labels,
            "protein_id": protein_ids
        }

    return collate_fn


In [8]:
def compute_pos_weight_from_loader(train_loader, num_labels, device, max_batches=None, clip_max=50.0):
    """
    Compute pos_weight for BCEWithLogitsLoss:
      pos_weight[j] = (neg_j / (pos_j + eps))
    Uses batches from train_loader (optionally limited).
    """
    pos = np.zeros(num_labels, dtype=np.float64)
    total = 0

    for bi, batch in enumerate(train_loader):
        y = batch["labels"].numpy()  # labels come from collate_fn on CPU
        pos += y.sum(axis=0)
        total += y.shape[0]
        if max_batches is not None and (bi + 1) >= max_batches:
            break

    eps = 1e-12
    neg = total - pos
    pw = neg / (pos + eps)

    # Clip to avoid extreme weights and instability
    pw = np.clip(pw, 1.0, clip_max).astype(np.float32)
    return torch.tensor(pw, dtype=torch.float32, device=device)

In [9]:
# ============================================================
# 4) Model: ESM2 encoder + mean pooling + linear head
# ============================================================
class EsmMultiLabel(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_labels)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        x = out.last_hidden_state                         # [B, L, H]
        mask = attention_mask.unsqueeze(-1).type_as(x)     # [B, L, 1]

        pooled = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)  # mean pool
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)                   # [B, num_labels]
        return logits



In [10]:

# ============================================================
# 5) Fine-tuning function (AMP + grad clip + optional freezing)
# ============================================================

THRESHOLD_GRID = np.concatenate([
    np.arange(0.001, 0.051, 0.005),
    np.arange(0.05, 0.53, 0.02)
])

def finetune_esm2_cafa6_discriminative(
    term_type="BP",
    num_labels=1000,
    max_length=1022,
    train_size=0.8,
    n_epochs=8,
    freeze_encoder_epochs=1,
    grad_clip=1.0,
    out_dir="/kaggle/working/esm2_finetune",
    obo_path=None,
    early_stopping_patience=3,
    metrics_every=1,
    encoder_lr=1e-6,
    head_lr=1e-4,
    pos_weight_clip=50.0
):
    os.makedirs(out_dir, exist_ok=True)
    device = CFG.device
    print("Device:", device)

    if obo_path is None:
        raise ValueError("Provide obo_path (CFG.OBO_PATH).")

    # Tokenizer/collate
    tokenizer = AutoTokenizer.from_pretrained(CFG.ESM2_MODEL_NAME)
    collate_fn = make_collate_fn(tokenizer)

    # Dataset (uses train_terms.tsv internally; ensure FASTA IDs normalized in the dataset)
    dataset = GOFineTuneDatasetFromTSV(
        fasta_path=CFG.TRAIN_SEQUENCES_PATH,
        tokenizer=tokenizer,
        train_terms_path=CFG.TRAIN_TERMS_PATH,
        term_type=term_type,
        num_labels=num_labels,
        max_length=max_length
    )

    # Terms list directly from TSV to match head order
    go_terms_list = get_top_terms_from_train_terms(CFG.TRAIN_TERMS_PATH, term_type, num_labels)

    # Build evaluator + ancestor matrix for metrics (and later closure)
    ia_df = pd.read_csv(CFG.IA_PATH, sep="\t", header=None).iloc[:, :2]
    ia_df.columns = ["term", "ia"]
    ia_weights = dict(zip(ia_df["term"].astype(str), ia_df["ia"].astype(float)))

    ancestors = load_go_ancestors(obo_path)
    A = build_ancestor_matrix(go_terms_list, ancestors)
    evaluator = Evaluator(ia_weights=ia_weights, go_terms=go_terms_list, ancestor_matrix=A)

    # Split
    train_len = int(len(dataset) * train_size)
    val_len = len(dataset) - train_len
    train_set, val_set = random_split(dataset, [train_len, val_len])

    train_loader = DataLoader(train_set, batch_size=CFG.batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader   = DataLoader(val_set, batch_size=CFG.batch_size, shuffle=False, collate_fn=collate_fn)

    # Model
    model = EsmMultiLabel(CFG.ESM2_MODEL_NAME, num_labels=num_labels, dropout=0.1).to(device)

    # Optional freeze encoder initially
    if freeze_encoder_epochs > 0:
        for p in model.encoder.parameters():
            p.requires_grad = False

    # Compute pos_weight from train_loader (fast estimate)
    pw_batches = getattr(CFG, "DEBUG_POS_WEIGHT_BATCHES", None)
    pos_weight = compute_pos_weight_from_loader(
        train_loader=train_loader,
        num_labels=num_labels,
        device=device,
        max_batches=pw_batches,       # None => full estimate
        clip_max=pos_weight_clip
    )
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    # Discriminative LR: separate param groups
    optimizer = torch.optim.AdamW([
        {"params": model.encoder.parameters(), "lr": encoder_lr},
        {"params": model.classifier.parameters(), "lr": head_lr},
    ])

    use_amp = (device.type == "cuda")
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    print(f"AMP enabled: {use_amp} | encoder_lr={encoder_lr:g} | head_lr={head_lr:g}")

    # Track best by IA-Fmax
    best_ia_fmax = -1.0
    best_threshold = 0.5
    best_state = None
    no_improve_epochs = 0

    for epoch in range(n_epochs):
        if epoch == freeze_encoder_epochs:
            # Unfreeze encoder + reset optimizer (still discriminative LR)
            for p in model.encoder.parameters():
                p.requires_grad = True
            optimizer = torch.optim.AdamW([
                {"params": model.encoder.parameters(), "lr": encoder_lr},
                {"params": model.classifier.parameters(), "lr": head_lr},
            ])
            print("Unfroze encoder parameters.")

        # ----- TRAIN -----
        model.train()
        running = 0.0
        train_bar = tqdm(train_loader, desc=f"[{term_type}] Train {epoch+1}/{n_epochs}", leave=False)

        for batch in train_bar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device).float()

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda', enabled=use_amp):
                logits = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(logits, labels)

            if not torch.isfinite(loss):
                raise ValueError("Non-finite loss during fine-tuning")

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
            scaler.step(optimizer)
            scaler.update()

            running += loss.item()
            train_bar.set_postfix(loss=f"{loss.item():.4f}")

        train_loss = running / max(len(train_loader), 1)

        # ----- VALIDATE -----
        model.eval()
        vloss = 0.0
        val_probs_list = []
        val_targets_list = []

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"[{term_type}] Val {epoch+1}/{n_epochs}", leave=False):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device).float()

                with torch.amp.autocast('cuda', enabled=use_amp):
                    logits = model(input_ids=input_ids, attention_mask=attention_mask)
                    loss = criterion(logits, labels)

                vloss += loss.item()

                probs = torch.sigmoid(logits).float().cpu().numpy()
                val_probs_list.append(probs)
                val_targets_list.append(labels.cpu().numpy())

        val_loss = vloss / max(len(val_loader), 1)
        val_probs = np.vstack(val_probs_list)
        val_targets = np.vstack(val_targets_list)

        # Guard
        val_probs = np.nan_to_num(val_probs, nan=0.0, posinf=1.0, neginf=0.0)
        val_targets = np.nan_to_num(val_targets, nan=0.0, posinf=0.0, neginf=0.0)

        print(f"[{term_type}] Epoch {epoch+1}/{n_epochs} | train loss: {train_loss:.4f} | val loss: {val_loss:.4f}")

        # ----- METRICS + threshold search -----
        do_metrics = ((epoch + 1) % metrics_every == 0) or (epoch == n_epochs - 1)
        thresholds = getattr(CFG, "DEBUG_THRESHOLD_GRID", None)
        if thresholds is None:
            thresholds = THRESHOLD_GRID

        #print("val_probs min/max:", val_probs.min(), val_probs.max(), "mean:", val_probs.mean())

        if do_metrics:
            t_f, fmax = evaluator.fmax(y_true=val_targets, y_prob=val_probs, thresholds=thresholds)
            t_ia, ia_fmax = evaluator.ia_fmax(y_true=val_targets, y_prob=val_probs, thresholds=thresholds)

            print(
                f"[{term_type}] "
                f"Fmax: {fmax:.4f} @ t={t_f:.3f} | "
                f"IA-Fmax: {ia_fmax:.4f} @ t={t_ia:.3f} "
            )

            if ia_fmax > best_ia_fmax:
                best_ia_fmax = ia_fmax
                best_threshold = t_ia
                best_state = copy.deepcopy(model.state_dict())
                no_improve_epochs = 0

                ckpt_path = os.path.join(out_dir, f"best_esm2_{term_type}_top{num_labels}.pt")
                torch.save(best_state, ckpt_path)
                print("✅ Saved best checkpoint:", ckpt_path)
            else:
                no_improve_epochs += 1

            if no_improve_epochs >= early_stopping_patience:
                print(f"[{term_type}] Early stopping: no IA-Fmax improvement for {early_stopping_patience} checks.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"[{term_type}] Best IA-Fmax: {best_ia_fmax:.4f} @ t={best_threshold:.3f}")
    return model, tokenizer, go_terms_list, A, best_threshold, best_ia_fmax


In [11]:
def apply_hierarchy_closure_probs(probs, A_bool):
    """
    probs: (N, T) in [0,1]
    A_bool: (T, T) boolean where A[child, ancestor] = True
    Returns probs_closed where ancestor prob >= max child prob.
    """
    if A_bool is None:
        return probs

    A = A_bool.astype(bool)
    out = probs.copy()

    # For each ancestor term j, find children indices where A[child, j] is True
    # Then out[:, j] = max(out[:, j], max(out[:, children], axis=1))
    for j in range(A.shape[1]):
        children = np.where(A[:, j])[0]
        if children.size == 0:
            continue
        out[:, j] = np.maximum(out[:, j], out[:, children].max(axis=1))

    # keep in [0,1]
    return np.clip(out, 0.0, 1.0)


In [12]:
# ============================================================
# 6) Predict on test FASTA (after fine-tuning)
# ============================================================
class TestFastaDataset(Dataset):
    def __init__(self, fasta_path, tokenizer, max_length=1022):
        self.records = list(SeqIO.parse(fasta_path, "fasta"))
        if hasattr(CFG, "DEBUG_TEST_N") and CFG.DEBUG_TEST_N is not None:
            self.records = self.records[:CFG.DEBUG_TEST_N]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        entry_id = rec.id
        seq = str(rec.seq)[: self.max_length]

        enc = self.tokenizer(
            seq,
            return_tensors="pt",
            padding=False,
            truncation=True,
            max_length=self.max_length
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "protein_id": entry_id
        }

def predict_test_probs(model, tokenizer, max_length=1022, batch_size=16):
    device = CFG.device
    collate = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

    def collate_test(batch):
        protein_ids = [x["protein_id"] for x in batch]
        batch_wo = [{"input_ids": x["input_ids"], "attention_mask": x["attention_mask"]} for x in batch]
        padded = collate(batch_wo)
        return {
            "input_ids": padded["input_ids"],
            "attention_mask": padded["attention_mask"],
            "protein_id": protein_ids
        }

    test_ds = TestFastaDataset(CFG.TEST_SEQUENCES_PATH, tokenizer, max_length=max_length)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_test)

    model.eval()
    all_probs = []
    all_ids = []

    use_amp = (device.type == "cuda")
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Predict test", leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            with torch.amp.autocast('cuda', enabled=use_amp):
                logits = model(input_ids=input_ids, attention_mask=attention_mask)

            probs = torch.sigmoid(logits).float().cpu().numpy()
            all_probs.append(probs)
            all_ids.extend(batch["protein_id"])

    return np.vstack(all_probs), all_ids



In [ ]:
# DEBUG = True
DEBUG = False

if DEBUG:
    print(">>> DEBUG MODE (quick smoke test)")

    CFG.DEBUG_TRAIN_N = 4000          # use first N train proteins
    CFG.DEBUG_TEST_N = 200            # predict first N test proteins
    CFG.DEBUG_POS_WEIGHT_BATCHES = 50 # estimate pos_weight from first N batches
    CFG.DEBUG_THRESHOLD_GRID = np.linspace(0.01, 0.50, 25)

    MAX_LEN = 256
    EPOCHS = 2

else:
    print(">>> FULL TRAIN MODE")

    CFG.DEBUG_TRAIN_N = None
    CFG.DEBUG_TEST_N = None
    CFG.DEBUG_POS_WEIGHT_BATCHES = None
    CFG.DEBUG_THRESHOLD_GRID = None

    MAX_LEN = 1022
    EPOCHS = 7   # or 10–15 with early stopping

OUT_DIR = "/kaggle/working/esm2_multi_ont"

# Discriminative LR defaults (good starting point)
ENC_LR = 1e-6
HEAD_LR = 1e-4

runs = [
    ("BP", 1000),
    ("MF", 500),
    ("CC", 500),
]

all_rows = []

for term_type, num_labels in runs:
    print("\n" + "="*60)
    print(f"Running {term_type} with top{num_labels}")
    print("="*60)

    model, tokenizer, go_terms_list, A, best_t, best_ia = finetune_esm2_cafa6_discriminative(
        term_type=term_type,
        num_labels=num_labels,
        max_length=MAX_LEN,
        n_epochs=EPOCHS,
        freeze_encoder_epochs=1,
        grad_clip=1.0,
        out_dir=OUT_DIR,
        obo_path=CFG.OBO_PATH,
        early_stopping_patience=4,
        metrics_every=1,
        encoder_lr=ENC_LR,
        head_lr=HEAD_LR,
        pos_weight_clip=50.0
    )

    print(f"[{term_type}] Using threshold {best_t:.3f} (best IA-Fmax {best_ia:.4f})")

    # Predict test
    test_probs, test_ids = predict_test_probs(
        model=model,
        tokenizer=tokenizer,
        max_length=MAX_LEN,
        batch_size=CFG.batch_size
    )
    print(term_type, "TEST probs min/max:", test_probs.min(), test_probs.max())

    # Hierarchy closure on probabilities
    test_probs = apply_hierarchy_closure_probs(test_probs.astype(np.float32), A)
    print(term_type, "AFTER closure min/max:", test_probs.min(), test_probs.max())

    # Convert to submission rows using per-ontology best threshold
    for i, pid in enumerate(test_ids):
        probs = test_probs[i]
        idxs = np.where(probs >= best_t)[0]
        for j in idxs:
            all_rows.append((pid, go_terms_list[j], float(probs[j])))

# Build submission dataframe
submission = pd.DataFrame(all_rows, columns=["proteinID", "GO_termID", "confidence"])

# If too strict (rare), fallback to top-N per protein PER ONTOLOGY is more complex;
# simplest fallback: keep top-20 overall per protein from the merged rows is not possible here.
# Usually thresholds won't be empty once trained. We'll only fallback if completely empty:
if submission.empty:
    raise ValueError("Submission is empty. Thresholds likely too strict; lower thresholds or use top-N.")

out_path = "/kaggle/working/submission.tsv"
submission.to_csv(out_path, sep="\t", index=False)
print("Saved:", out_path)
print("Rows:", len(submission))
submission.head(10)


>>> FULL TRAIN MODE

Running BP with top1000
Device: cuda


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


AMP enabled: True | encoder_lr=1e-06 | head_lr=0.0001


[BP] Train 1/7:  22%|██▏       | 900/4121 [02:05<06:52,  7.82it/s, loss=0.3092]